# Calibration-target ladder — how strongly does a fixed target assumption bias the inferred parameters? (Reviewer 1, R1-C6)

<details>
<summary>Every lane starts from the same state and drives to the same targets — except one, which is a ladder.</summary>

Reviewer 1 notes that total blood volume (TBV) and its distribution across compartments are
fixed to a common reference for all cohorts, and asks for *"some quantification of how
strongly this assumption biases the inferred shock parameter distributions"*.

This notebook is that quantification. It runs batched populations in which **every lane is
identical** — same initial state, same sixteen calibration targets — **except one target, which
is stepped from `ladder.min` to `ladder.max` times its reference value**. Each lane is then
recalibrated by the unchanged EFC procedure, and the converged parameter set is read at run end.
How far a parameter travels along that step *is* its sensitivity to the fixed-target assumption.

The laddered target is **any** observation EFC calibrates — a compartment volume, a pressure, an
amplitude, a stroke volume — because every calibration entry holds its setpoint in a
`target_<observation>` state and `batchedCalibration` injects state columns per lane. R1-C6 is
the volume instance of that machinery; `ladder.targets` selects which ones to sweep.

Two ladder modes, selected by `ladder.modes`:

- **`partition`** — ladder one calibration target at a time (one ladder per entry of
  `ladder.targets`). Every other target, and the total loop volume, are unchanged; for a volume
  target the uncontrolled systemic-venous compartment absorbs the complement. This is the
  reviewer's *"distribution across compartments"* degree of freedom when the listed targets are
  volumes, and a general target-sensitivity sweep otherwise.
- **`tbv`** — volume-only by construction: every compartment's initial volume *and* its volume
  target are scaled together, holding the reference partition. This is the reviewer's *"total
  blood volume"* degree of freedom, and it is the one the published population never actually
  exercised: `batchedCalibration` injects only the `target_*` columns per lane, so the physical
  `V_*` states always came from the scenario-level `TotalBloodVolume`.

Because $\ln y^{*} = \ln y_{ref} + \ln f$, the slope of $\ln p$ against the ladder fraction
$f$ is exactly the elasticity $\partial\ln p / \partial\ln y^{*}$: a parameter with
$\varepsilon = 0.8$ is biased by about 8 % for every 10 % error in the assumed target.

Untargeted observables are excluded from every residual by
`library.run.progress.obsTargetArrays` — notably `V_Vs` (the slack compartment, which no
controller drives) and `Cyc_HC` (a driver input). They are therefore not laddereable either, and
are dropped from `ladder.targets` with a warning. `V_Vs` is instead read out on its own as the
slack-absorption panel.
</details>

In [ ]:
# region -> runConfig — the single run-configuration surface (device/precision applied before JAX)
# The ONE place run configuration lives (project rule: repo CLAUDE.md). Defined first so the
# device/precision block applies before JAX initialises in the Imports cell.
runConfig = {
    # --- file references ---
    "model":    "cvModel_linear.json",   # cubic controllers — the stack the published population used
    "scenario": "sepsis_linear.json",    # twin targets + calibration stages + convergence obs list
    "mode":     "calibration",    # staged calibration to the (per-lane laddered) twin targets

    # --- pipeline phases (run + plot separable) ---
    "run":  False,    # phase 1 — run each ladder mode, save one artifact per mode.
                     #   Once the artifacts exist under output.path, flip False and re-run
                     #   phase 2 alone (~80 lanes of staged calibration, ~15 min CPU).
    "plot": True,    # phase 2 — load + analyse + elasticity + figure + table

    # --- device / precision (applied in Imports cell, before `import jax`) ---
    "device": {
        "useGpu":    False,       # GPU is OFF by default everywhere (repo HARD RULE)
        "precision": "float64",   # "float64" or "float32"
    },

    # --- volume ladder (NEW knob) — the axis that distinguishes the lanes ---
    "ladder": {
        "modes":   ["partition", "tbv"],   # which ladders to run; one artifact each
        "targets": ["avg_P_Vs",
                    "avg_P_Cp",
                    "avg_P_Cs",
                    "keep_max_P_As",
                    "keep_SV_Hl",
                    "keep_max_P_Ap",
                    "amp_P_As",
                    "amp_P_Ap",
                    "avg_V_As",
                    "avg_V_Ap",
                    "avg_V_Hl",
                    "avg_V_Hr",
                    "avg_V_Cs",
                    "avg_V_Vt",
                    "avg_V_Cp",
                    "avg_V_Vp"
                    ],
                                  # partition mode: one ladder per listed observation. ANY
                                  #   calibrated observation works (volume, pressure, amplitude,
                                  #   stroke volume) — the rung reference is its twin target and
                                  #   the injected column is its target_* setpoint state. Entries
                                  #   with no controller or no twin target (V_Vs, Cyc_HC) are
                                  #   dropped with a warning. R1-C6 = the avg_V_* subset.
        "rungs":   32,            # lanes per ladder
        "min":     0.75,           # ladder low  — FRACTION of the compartment reference volume
        "max":     1.25,           # ladder high — FRACTION of the compartment reference volume
        "spacing": "linear",      # "linear" or "geometric"
        "clampHeadroom": True,    # widen the clamp of every laddered target's controlling
                                  #   parameter by the ladder span so the extreme rungs are
                                  #   reachable (the clamp is baked per run, not per lane; V0_*
                                  #   is sized off the reference target since stateSetup
                                  #   re-derives its max from the reference compartment volume)
    },

    # --- batched solve ---
    "solver":    {"type": "euler"},  # "euler" or "rk4"; keep euler (dt == step) on the integrated stack
    "chunkSize": 64,                 # samples per vmap (VRAM bound)
    "calibration": {},               # per-key merge over scenario calibration ({} = as-is)

    # --- analysis ---
    "analysis": {
        "atm":             760.0,     # atmospheric offset for absolute-pressure signals
        "divergenceLimit": 50000.0,   # |value| >= this in any obs/param -> BAD lane
        "errorTarget":     0.5,       # SUCCESS if max |rel err| (%) <= this
        "nParamsInTable":  2,         # most-biased calibrated params written into the LaTeX table
        "saturationTol":   1e-3,  # SATURATED CONTROLLER: a calibrated parameter this close
                                  #   (as a fraction of its clamp span) to either clamp bound
                                  #   at run end has run out of travel and stopped tracking
                                  #   its target -> the lane is dropped from every analysis
                                  #   below, like a diverged one. The clamp is soft, so a
                                  #   pinned parameter settles just off the bound, never on it.
    },

    "progressEvery": 0,     # live convergence line every N sim seconds (0 = off for the sweep)
    "saveRaw":       True,  # stream every lane's trajectory (Phase 2 reads run-end from `raw`)

    # --- output (one artifact per ladder mode) ---
    "output": {
        "save":  True,
        "path":  "notebookData/volume",
        "names": {
            "partition": "volume_ladder_partition.h5",
            "tbv":       "volume_ladder_tbv.h5",
        },
        "logProgress": True,
    },

    # --- paper artifact routing (guarded by paper.emit) ---
    "paper": {
        "emit":         True,
        "generatedDir": "EFC_Paper/responseLetter/generated",
        "imagesDir":    "EFC_Paper/responseLetter/generated",
        "tableName":    "volumeLadder.tex",
        "figName":      "volumeLadder.png",
    },

    "postProcessing": None,
    "plots":          [],
    "plotOpts":       {"targetPoints": 1500},   # strided-decimation target for the convergence plots
    "printStatus":    True,
    "printEveryPct":  100,

    # --- integration numerics (override scenario shared.integration) ---
    "runTime": 10,       # simulated seconds per internal run
    "dt":      0.0005,   # integrator step (== cycle step on the integrated stack)
    "dtDense": 1.0,      # save grid: 1 Hz run-end sampling is enough for the residual read
}
# endregion

## Imports

In [ ]:
# region -> imports + device/precision (must precede `import jax`)
# ---- repo-root bootstrap: run from any cwd (make `library` importable + resolve the relative
# ---- notebookData/ + config/ paths). Walks up to the dir containing library/. ----
import os, sys
_root = os.path.abspath(os.getcwd())
while not os.path.isdir(os.path.join(_root, "library")) and _root != os.path.dirname(_root):
    _root = os.path.dirname(_root)
if _root not in sys.path:
    sys.path.insert(0, _root)
os.chdir(_root)

# ---- device / precision (from runConfig, MUST run before JAX initialises) ----
useGpu    = runConfig["device"]["useGpu"]
precision = runConfig["device"]["precision"]

if useGpu:
    os.environ.pop("CUDA_VISIBLE_DEVICES", None)
    os.environ["JAX_PLATFORMS"] = "cuda"
    os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
    os.environ["XLA_PYTHON_CLIENT_ALLOCATOR"]   = "platform"
else:
    os.environ["CUDA_VISIBLE_DEVICES"] = "-1"
    os.environ["JAX_PLATFORMS"] = "cpu"

import jax
jax.config.update("jax_enable_x64", precision == "float64")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import json, time

import library.run.runner as runner                    # buildSimulationParams
import library.run.runnerBatchSI as runnerBatchSI       # batched (vmapped) SI calibration
import library.run.progress as progressLib              # obsTargetArrays (the ONE target map)
import library.viz.plots as libPlots                    # plotVolumeLadder / plotCalibrationConvergence
import library.utils as utils
from library.hdf5 import schema_pop                       # population artifact
from library.hdf5.raw_stream import RawTraceStreamWriter  # async raw streaming writer
import library.postproc.reporting as reporting            # shared scope / rejection report

np.set_printoptions(suppress=True)
print("devices:", jax.devices(), "| x64:", jax.config.jax_enable_x64)
# endregion

## Assemble `simulationParams` + resolve the reference targets

<details>
<summary>Load the twin + convergence observation list, resolve each ladder target's reference and controller, and specialise runConfig per mode.</summary>

`progressLib.obsTargetArrays` is the single "where should this observable land" map, so the
reference rung of every ladder is the twin target itself — `TotalBloodVolume x
volumeDistribution[c]` for a compartment volume, the twin pressure/amplitude/stroke-volume entry
for everything else. The ladder is expressed as a *fraction* of that reference, so one
`[min, max]` range is meaningful for every target regardless of its units or size.

Which observations can be laddered is read off the model's `calibration` block rather than
hard-coded: an entry with a `targetValue` param holds its setpoint in that named state, which is
exactly the seam `batchedCalibration` injects per lane. A `ladder.targets` entry with no such
controller (or no twin target — `V_Vs`, `Cyc_HC`) is dropped with a warning. The same block
gives the **controlling parameter** of each target, which is what `clampHeadroom` widens.

Which compartments are targeted is likewise read off `varTarget` (the `avg_V_*` entries);
whatever is left over in `volumeDistribution` is the **slack** compartment, which no controller
drives and which therefore absorbs any volume the targeted compartments do not claim.

`cfgForMode` is the single place a ladder mode turns into a run configuration: it picks the
mode's artifact name out of `output.names` and, when `clampHeadroom` is on, widens the clamp of
every laddered target's controlling parameter through `calibration.bounds` —
`buildSimulationParams` merges it and `stateSetup` applies it *after* the TBV-derived assignment
and *before* the constants are baked: a plain per-key override, not a second mechanism.
</details>

In [ ]:
# region -> assemble simulationParams + resolve reference targets, controllers and clamp headroom
scenario  = utils.loadScenario(runConfig["scenario"])
modelJSON = utils.loadJSONfile(utils.configPath("models", runConfig["model"]))

twin    = scenario["shared"]["twin"]["twinTargets"]
volDist = scenario["shared"]["twin"]["volumeDistribution"]
ladder  = runConfig["ladder"]
ATM     = runConfig["analysis"]["atm"]

observations = scenario["convergence"]["observations"]

def obsOffset(name):
    """Atmospheric offset baked into absolute-pressure signals (gauge = raw - offset)."""
    return utils.obsOffset(name, ATM)

# The ONE twin -> target map, in gauge units (NaN = untargeted: V_Vs slack, Cyc_HC driver input).
targetArr, offsetArr = progressLib.obsTargetArrays(observations, twin, volDist, ATM)
obsIdx   = {o: i for i, o in enumerate(observations)}
targeted = ~np.isnan(targetArr)

# --- reference volume partition (the `tbv` ladder) + which compartments carry a controller ---
TBV_REF = twin["TotalBloodVolume"]
refVol  = {c: TBV_REF * f for c, f in volDist.items()}            # reference volume per compartment (mL)
targetedComps = [c for c in volDist
                 if any(e["params"].get("varTarget") == "avg_V_" + c
                        for e in modelJSON["calibration"].values())]
slackComps    = [c for c in volDist if c not in targetedComps]    # no controller -> absorbs the rest
SLACK         = "V_" + slackComps[0] if slackComps else None

# --- observation -> (setpoint state, controlling parameter) ---------------------------------
# Every state-type calibration entry drives one observable (`varTarget`) through one parameter
# (the calibration key) and holds its setpoint in the state named by `targetValue`. That state is
# the seam `batchedCalibration` overwrites per lane, so ANY observation appearing here can be
# laddered — volume, pressure, amplitude or stroke volume alike.
stateForObs = {v["params"]["varTarget"]: v["params"]["targetValue"]
               for v in modelJSON["calibration"].values() if "targetValue" in v["params"]}
paramForObs = {v["params"]["varTarget"]: k for k, v in modelJSON["calibration"].items()
               if "targetValue" in v["params"]}
obsForState = {s: o for o, s in stateForObs.items()}

# --- validate ladder.targets: keep what has a twin target AND a setpoint state ---------------
ladderTargets, droppedTargets = [], []
for o in ladder["targets"]:
    if o not in obsIdx:
        droppedTargets.append((o, "not in scenario convergence observations"))
    elif not targeted[obsIdx[o]]:
        droppedTargets.append((o, "untargeted (no twin target — excluded from every residual)"))
    elif o not in stateForObs:
        droppedTargets.append((o, "no state-type calibration controller (no target_* state)"))
    else:
        ladderTargets.append(o)
refTarget = {o: float(targetArr[obsIdx[o]]) for o in ladderTargets}   # ladder rung 1.0, gauge units

# --- the ladder rungs (shared by every ladder; a FRACTION of each reference value) ----------
fracs = (np.geomspace(ladder["min"], ladder["max"], ladder["rungs"])
         if ladder.get("spacing", "linear") == "geometric"
         else np.linspace(ladder["min"], ladder["max"], ladder["rungs"]))

outDir = runConfig["output"]["path"]
paths  = {m: os.path.join(outDir, runConfig["output"]["names"][m]) for m in ladder["modes"]}

def headroomBounds(ladderedObs):
    """[min, max] override for the controlling parameter of every laddered target.

    A target stepped to `ladder.max` x reference is only reachable if its controlling parameter
    can travel that far, so each clamp is widened by the ladder span. `V0_*` is the special case:
    `stateSetup._configureCalibration` re-derives its maxValue from the reference compartment
    volume (ignoring the model JSON), so its headroom is sized off the laddered reference itself.

    NOTE: bounds are baked per RUN, not per lane, so a clampHeadroom run solves its reference rung
    in a slightly wider box than the published population; set clampHeadroom False to reproduce it."""
    loF, hiF = min(1.0, ladder["min"]), max(1.0, ladder["max"])
    bounds = {}
    for o in ladderedObs:
        p = paramForObs.get(o)
        if p is None:
            continue
        prm = modelJSON["calibration"][p]["params"]
        lo, hi = float(prm["minValue"]), float(prm["maxValue"])
        bounds[p] = [lo * loF, (refTarget.get(o, hi) if p.startswith("V0_") else hi) * hiF]
    return bounds

def cfgForMode(mode):
    """runConfig specialised to one ladder mode: its artifact name, plus — when clampHeadroom is
    on — widened bounds for the controlling parameter of every target that mode ladders."""
    cfg = {**runConfig, "output": {**runConfig["output"],
                                   "name": runConfig["output"]["names"][mode]}}
    if ladder.get("clampHeadroom", False):
        laddered = ladderTargets if mode == "partition" else ["avg_V_" + c for c in targetedComps]
        cal = runConfig.get("calibration", {})
        cfg["calibration"] = {**cal, "bounds": {**cal.get("bounds", {}),
                                                **headroomBounds(laddered)}}
    return cfg

simulationParams = runner.buildSimulationParams(cfgForMode(ladder["modes"][0]), scenario)
calConf      = simulationParams["simulationConf"]["calibration"]
calibParams  = calConf["adaptive"]["parameters"]     # the 16 calibrated params (run-end read)

print(f"TBV reference = {TBV_REF} mL | targeted compartments = {targetedComps} | slack = {SLACK}")
print(f"ladder rungs = {ladder['rungs']} in [{ladder['min']}, {ladder['max']}] x reference "
      f"({ladder.get('spacing', 'linear')}) | modes = {ladder['modes']}")
for o, why in droppedTargets:
    print(f"  [skip] ladder target {o!r}: {why}")
for m, p in paths.items():
    print(f"  {m:<10} -> {p}")
_hb = headroomBounds(ladderTargets) if ladder.get("clampHeadroom", False) else {}
pd.DataFrame({"ladder target": ladderTargets,
              "reference": [refTarget[o] for o in ladderTargets],
              "setpoint state": [stateForObs[o] for o in ladderTargets],
              "controlling param": [paramForObs[o] for o in ladderTargets],
              "clamp (headroom)": [_hb.get(paramForObs[o], "—") for o in ladderTargets]})
# endregion

## Reference twin targets + atmospheric offsets

<details>
<summary>The reference target for every observation — untargeted observables are NaN and drop out of every residual.</summary>

`progressLib.obsTargetArrays` (called in the cell above) is the single definition of "where
should this observable land", shared with the in-solve progress line and the MCMC baselines. It
deliberately returns NaN for `V_Vs` (the slack compartment — no controller targets it) and
`Cyc_HC` (the cycle period, a driver input): both are excluded from every residual here, and both
are therefore refused as ladder targets. The `ladder` column marks the observations this run
steps.
</details>

In [ ]:
# region -> reference target table: twin target, atmospheric offset and ladder membership per obs
print(f"{targeted.sum()}/{len(observations)} observations are calibration targets; "
      f"untargeted: {[o for o, t in zip(observations, targeted) if not t]}")
print(f"laddered: {ladderTargets}")
pd.DataFrame({"observation": observations, "target": targetArr, "offset": offsetArr,
              "ladder": [o in ladderTargets for o in observations]})
# endregion

## Ladder axis — the per-lane injection matrix

<details>
<summary>Build (sampled, param_names, laneLadder, laneFrac) per mode; the lanes differ only in the laddered column.</summary>

`batchedCalibration` varies lanes by overwriting `Y0` columns named by `param_names` — and any
state name is fair game, both the `target_*` setpoint states and the physical `V_*` compartment
states. That single seam carries both modes:

- **`partition`**: `param_names` are the setpoint states of every validated ladder target
  (`stateForObs`), whatever their units. Every lane is injected with the full reference row, then
  one column is stepped across its ladder. Making the shared baseline explicit (rather than
  leaning on the scalar prep) costs nothing and keeps the artifact self-describing.
- **`tbv`**: `param_names` are the targeted `target_avg_V_*` states **plus** all nine physical
  `V_*` states. Injecting the `V_*` states is what actually moves total blood volume; injecting
  the targets alongside keeps the setpoints consistent with the new total. Pressure and flow
  targets stay at reference, isolating the volume degree of freedom.

This cell is deterministic and runs in both phases, so Phase 2 can reconstruct which ladder and
which rung each lane belongs to without re-reading the artifact.
</details>

In [ ]:
# region -> ladder axis: per-mode (sampled, param_names, laneLadder, laneFrac) injection matrices
def buildLadder(mode):
    """(sampled (N,P), names (P,), laneLadder (N,), laneFrac (N,)) for one ladder mode.

    Lanes are identical except the laddered column: `laneLadder[i]` names the ladder lane i
    belongs to and `laneFrac[i]` is its rung as a fraction of the reference value."""
    R = ladder["rungs"]
    if mode == "partition":
        obs   = list(ladderTargets)
        names = [stateForObs[o] for o in obs]
        ref   = np.array([refTarget[o] for o in obs])             # twin target = rung 1.0
        sampled = np.tile(ref, (R * len(obs), 1))                 # every lane at reference ...
        for k in range(len(obs)):
            sampled[k * R:(k + 1) * R, k] = ref[k] * fracs        # ... then one column laddered
        laneLadder = np.repeat(np.array(obs, dtype=object), R)
        laneFrac   = np.tile(fracs, len(obs))
    elif mode == "tbv":
        names = ([stateForObs["avg_V_" + c] for c in targetedComps] + ["V_" + c for c in volDist])
        tbv   = TBV_REF * fracs                                    # total scaled, partition held
        sampled = np.column_stack([tbv * volDist[c] for c in targetedComps]
                                  + [tbv * volDist[c] for c in volDist])
        laneLadder = np.array(["TBV"] * R, dtype=object)
        laneFrac   = fracs.copy()
    else:
        raise ValueError(f"unknown ladder mode {mode!r} (expected 'partition' or 'tbv')")
    return sampled, names, laneLadder, laneFrac

ladderAxes = {m: buildLadder(m) for m in ladder["modes"]}
for m, (s, n, ll, lf) in ladderAxes.items():
    print(f"[{m}] lanes={s.shape[0]} injected columns={s.shape[1]} "
          f"ladders={list(dict.fromkeys(ll))}")
pd.DataFrame({"rung": range(len(fracs)), "fraction": fracs,
              "TBV_mL (tbv mode)": TBV_REF * fracs}).T
# endregion

## Run — one batched calibration per ladder mode

<details>
<summary>One `batchedCalibration` per ladder mode, each streamed to its own population artifact.</summary>

The run helper is the standard population-notebook body — `prepare` → `rawLayout` →
`schema_pop.init_population` → `RawTraceStreamWriter` → `batchedCalibration` → final states,
timings and progress — driven once per mode with that mode's injection matrix and the
`cfgForMode` configuration assembled above. Nothing about the calibration method changes between
modes or between lanes; the only thing that varies is the `Y0` column the ladder writes.
</details>

In [ ]:
# region -> run each ladder mode as one batched calibration, save one population artifact per mode
def _runPop(mode):
    """Run one vmapped calibration population for `mode` and save it via schema_pop."""
    sampled, names, _, _ = ladderAxes[mode]
    cfg  = cfgForMode(mode)
    path = paths[mode]
    sp   = runner.buildSimulationParams(cfg, scenario)

    prep   = runnerBatchSI.prepare(sp)
    layout = runnerBatchSI.rawLayout(sp, prepared=prep)
    stateNames = layout["stateNames"]
    rawDtype   = "float64" if jax.config.jax_enable_x64 else "float32"
    N = sampled.shape[0]
    traceNames = list(dict.fromkeys(list(observations) + list(names)))
    saveRaw = runConfig.get("saveRaw", False) and runConfig["output"]["save"]

    if runConfig["output"]["save"]:
        os.makedirs(runConfig["output"]["path"], exist_ok=True)
        schema_pop.init_population(
            path, param_names=names, state_names=stateNames,
            observation_names=observations, sampled_params=sampled,
            model_structure=utils.modelStructureJSON(prep["modelStructure"]),
            problem={"names": names, "bounds": [], "num_vars": len(names)},
            conf=cfg, meta={"twinTargets": twin, "tag": mode})

    def rawWriterFactory(signalNames, totalPoints, time_, nDense):
        return RawTraceStreamWriter(path, N=N, signalNames=signalNames,
                                    totalPoints=totalPoints, nDense=nDense, time=time_, dtype=rawDtype)

    t0 = time.time()
    batch = runnerBatchSI.batchedCalibration(
        sp, sampled, names, observations,
        chunkSize=runConfig.get("chunkSize", 64), printStatus=runConfig.get("printStatus", True),
        printEveryPct=runConfig.get("printEveryPct"), traceNames=traceNames,
        rawWriterFactory=(rawWriterFactory if saveRaw else None), prepared=prep)
    wall = time.time() - t0
    print(f"[{mode}] {runConfig['model']}: N={N} solved in {wall:.1f}s (solver={sp['solver']['type']})")

    if runConfig["output"]["save"]:
        schema_pop.write_final_states(path, np.asarray(batch["finalStates"]),
                                      [str(i) for i in range(N)])
        schema_pop.write_timings(path, [wall], meta={
            "device": "gpu" if useGpu else "cpu", "precision": precision,
            "solver": sp["solver"]["type"], "dt": sp["dt"], "runTime": sp["runTime"],
            "nrModels": N, "stack": "SI", "total_wall": wall, "tag": mode})
        schema_pop.write_progress(path, batch.get("progress"), meta={
            "model": runConfig["model"], "scenario": runConfig["scenario"], "mode": runConfig["mode"],
            "solver": sp["solver"]["type"], "nrModels": N, "stack": "SI", "ladderMode": mode})
    return batch

if runConfig["run"]:
    for _m in ladder["modes"]:
        _ = _runPop(_m)
    print("run phase complete.")
# endregion

# Phase 2 — Load & analyse (from the saved files)

<details>
<summary>Reconstruct each mode's population and build its PER-LANE target matrix.</summary>

Unlike a single-axis sweep, the lanes here do not share one target vector: the laddered
observation's target *is* the laddered value. So each mode gets a `targetMatrix (N, nObs)` — the
reference row tiled, with every injected `target_*` column written back into its observation
slot. Every downstream error computation indexes that matrix, never `targetArr`.

Missing artifacts are skipped rather than fatal, so a partial Phase 1 still analyses.
</details>

In [ ]:
# region -> load each ladder artifact + build its per-lane target matrix
def _loadPop(path):
    """Return (obsMatrix (N,nObs) gauge, lastRow (N,C), sigIdx, sig, modelStructure)."""
    import h5py
    with h5py.File(path, "r") as f:
        sig     = list(f["raw_signal_names"].asstr()[:])
        ms      = json.loads(f["model_structure"].asstr()[()])
        src     = "raw_coarse" if "raw_coarse" in f else "raw"
        lastRow = np.asarray(f[src][:, -1, :])                 # (N, C) run-end values
    sigIdx = {n: i for i, n in enumerate(sig)}
    obsM = np.full((lastRow.shape[0], len(observations)), np.nan)
    for j, o in enumerate(observations):
        if o in sigIdx:
            obsM[:, j] = lastRow[:, sigIdx[o]] - obsOffset(o)
    return obsM, lastRow, sigIdx, sig, ms

if runConfig["plot"]:
    divLim = runConfig["analysis"]["divergenceLimit"]
    pops, modelStructure = {}, None

    for m in ladder["modes"]:
        path = paths[m]
        if not os.path.exists(path):
            print(f"[{m}] artifact missing ({path}) — skipped")
            continue
        sampled, names, laneLadder, laneFrac = ladderAxes[m]
        obsM, lastRow, sigIdx, sig, ms = _loadPop(path)
        N = lastRow.shape[0]

        # per-lane targets: reference row, with each injected target_* column written back
        tgtM = np.tile(targetArr, (N, 1))
        for j, nm in enumerate(names):
            o = obsForState.get(nm)                             # setpoint state -> its observation
            if o in obsIdx:
                tgtM[:, obsIdx[o]] = sampled[:, j]

        plotParams  = [p for p in calibParams if p in sigIdx]
        paramMatrix = (np.column_stack([lastRow[:, sigIdx[p]] for p in plotParams])
                       if plotParams else np.zeros((N, 0)))
        finite = schema_pop.good_run_mask(obsM, divLim, param_matrix=paramMatrix)

        # A controller that ends the run against its clamp has run out of travel: the parameter
        # is pinned and no longer tracks its target, so the lane is not a calibration result and
        # is dropped like a diverged one (it otherwise flattens the elasticity fit). Bounds are
        # read from the artifact's OWN model_structure — the resolved clamp the solve used,
        # `clampHeadroom` widening included — not from the model JSON.
        satTol    = runConfig["analysis"].get("saturationTol", 1e-3)
        saturated = np.zeros(N, dtype=bool)
        satBy     = {}
        for j, p in enumerate(plotParams):
            prm    = ms["calibration"].get(p, {}).get("params", {})
            lo, hi = prm.get("minValue"), prm.get("maxValue")
            if lo is None or hi is None:                         # e.g. t_Sys_HC (unclamped)
                continue
            edge = satTol * (hi - lo)
            hit  = (paramMatrix[:, j] <= lo + edge) | (paramMatrix[:, j] >= hi - edge)
            if hit.any():
                satBy[p] = int(hit.sum())
            saturated |= hit
        finite &= ~saturated

        rawObsM = obsM.copy()                                   # pre-mask copy for the scope report
        obsM[~finite] = np.nan
        wall, timingMeta = schema_pop.read_timings(path)

        pops[m] = {"path": path, "obsMatrix": obsM, "rawObs": rawObsM, "lastRow": lastRow,
                   "sigIdx": sigIdx, "targetMatrix": tgtM, "paramMatrix": paramMatrix,
                   "plotParams": plotParams, "finite": finite, "timingMeta": timingMeta,
                   "laneLadder": laneLadder, "laneFrac": laneFrac, "N": N,
                   "saturated": saturated,
                   "slack": (lastRow[:, sigIdx[SLACK]] if SLACK in sigIdx else np.full(N, np.nan))}
        modelStructure = modelStructure or ms
        print(f"[{m}] loaded {N} lanes from {path} | valid {finite.sum()}/{N} "
              f"| saturated dropped {int(saturated.sum())}"
              + (f" {satBy}" if satBy else ""))

    if not pops:
        print("no artifacts loaded — run phase 1 first (runConfig['run'] = True)")
# endregion

## Error summary

<details>
<summary>Per-observation relative / absolute error against each lane's OWN target, faulty lanes dropped.</summary>

Errors are scored against `targetMatrix`, so a laddered lane is judged against the target it was
actually asked to hit — not against the reference. A ladder that stays flat and low here is a
ladder the method absorbed; a ladder that climbs marks the region where the imposed partition
stops being compatible with the closed-loop equation system.
</details>

In [ ]:
# region -> error summary per ladder mode — drop faulty lanes, per-observation relative/absolute error
if runConfig["plot"] and pops:
    obsTargeted = [o for o, t in zip(observations, targeted) if t]
    errByMode = {}

    for m, P in pops.items():
        good = P["finite"]
        og   = P["obsMatrix"][good][:, targeted]
        tg   = P["targetMatrix"][good][:, targeted]
        errRel = (og - tg) / tg * 100.0
        errByMode[m] = errRel
        success = np.max(np.abs(errRel), axis=1) <= runConfig["analysis"]["errorTarget"]
        print(f"[{m}] {good.sum()}/{len(good)} valid lanes "
              f"({(~good).sum()} faulty/diverged dropped) | "
              f"within max|rel err| <= {runConfig['analysis']['errorTarget']}%: "
              f"{success.sum()}/{len(success)}")

    summary = pd.DataFrame({"observation": obsTargeted})
    for m, errRel in errByMode.items():
        summary[f"{m}_mean_rel_%"] = np.nanmean(errRel, axis=0)
        summary[f"{m}_max_rel_%"]  = np.nanmax(np.abs(errRel), axis=0)
    summary
# endregion

## Scope / rejection report

In [ ]:
# region -> scope / rejection report (why each lane was dropped)
if runConfig["plot"] and pops:
    for m, P in pops.items():
        print(f"--- {m} ---")
        _ = reporting.scopeRejectionReport(P["rawObs"], P["paramMatrix"], observations,
                                           P["plotParams"], lim=divLim)
# endregion

## Per-observation error distribution

In [ ]:
# region -> boxplot: relative error distribution per observation, one panel per ladder mode
if runConfig["plot"] and pops:
    ms = [m for m in pops if errByMode[m].size]
    if ms:
        fig, axs = plt.subplots(len(ms), 1, figsize=(12, 4.6 * len(ms)), squeeze=False)
        et = runConfig["analysis"]["errorTarget"]
        for ax, m in zip(axs[:, 0], ms):
            ax.boxplot(errByMode[m], tick_labels=utils.labelsFor(obsTargeted, "latex"),
                       showfliers=False)
            ax.axhline(0.0, color="k", lw=0.8)
            ax.axhline(et, color="r", ls="--", lw=0.8, label=f"±{et}% target")
            ax.axhline(-et, color="r", ls="--", lw=0.8)
            ax.set_ylabel("relative error (%)")
            ax.set_title(f"{m} ladder — convergence error across "
                         f"{pops[m]['finite'].sum()} valid lanes (all rungs)")
            ax.tick_params(axis="x", rotation=90)
            ax.legend()
        plt.tight_layout()
        plt.show()
# endregion

## Calibration convergence — observation traces vs target

<details>
<summary>Per-lane observation trajectory over the whole calibration, one line per rung, one figure per ladder.</summary>

Colour encodes the ladder fraction, so a fan of lines that all land on their own setpoint is a
ladder the method tracked cleanly. Targets are drawn from the reference row, so on the laddered
observation the lines deliberately spread away from the drawn target — that spread *is* the
ladder.
</details>

In [ ]:
# region -> calibration convergence: per-lane observation traces vs target (one line per rung)
run = False
if run == True:
    if runConfig["plot"] and pops:
        import h5py
        calib        = (modelStructure or {}).get("calibration", {})
        targetsByObs = {o: t for o, t in zip(observations, targetArr)}
        offsetsByObs = {o: off for o, off in zip(observations, offsetArr)}
        targetPoints = runConfig["plotOpts"]["targetPoints"]

        def _rawBlock(path, rows, wanted):
            """{name: (len(rows), T)} run traces + the time vector, decimated to plotOpts.targetPoints."""
            with h5py.File(path, "r") as f:
                if "raw_coarse" in f:
                    src, tkey, ds, scope = "raw_coarse", "raw_coarse_time", 1, "whole calibration"
                else:
                    src, tkey = "raw", "raw_time"
                    ds, scope = max(1, f["raw"].shape[1] // targetPoints), "whole calibration (full raw)"
                sigN  = list(f["raw_signal_names"].asstr()[:])
                block = f[src][rows, ::ds, :]
                tvec  = f[tkey][::ds]
            return {n: block[:, :, sigN.index(n)] for n in wanted if n in sigN}, tvec, scope

        for m, P in pops.items():
            for name in dict.fromkeys(P["laneLadder"]):
                rows = np.where((P["laneLadder"] == name) & P["finite"])[0]
                if rows.size == 0:
                    continue
                paramForObs = {calib[p]["params"]["varTarget"]: p
                            for p in P["plotParams"] if p in calib}
                tr, tvec, scope = _rawBlock(P["path"], rows, obsTargeted)
                libPlots.plotCalibrationConvergence(
                    tr, traceT=tvec, targets=targetsByObs, offsets=offsetsByObs,
                    paramForObs=paramForObs, runValues=P["laneFrac"][rows],
                    runValueLabel=f"{name} target / reference",
                    title=f"Calibration convergence ({scope}) — {m} ladder on {name}")
                plt.show()
# endregion

## Calibration convergence — calibrated parameters

<details>
<summary>Each calibrated parameter's trajectory over the whole calibration, one line per rung.</summary>

Where the observation panels show whether the ladder was *tracked*, these show the cost of
tracking it: how far each of the sixteen inferred parameters had to travel to satisfy a shifted
volume target. A parameter whose fan is wide is one whose inferred value is hostage to the
partition assumption.
</details>

In [ ]:
# region -> calibration convergence: one panel per calibrated parameter (one line per rung)
if run == True:
    if runConfig["plot"] and pops:
        for m, P in pops.items():
            for name in dict.fromkeys(P["laneLadder"]):
                rows = np.where((P["laneLadder"] == name) & P["finite"])[0]
                if rows.size == 0:
                    continue
                tr, tvec, scope = _rawBlock(P["path"], rows, P["plotParams"])
                libPlots.plotCalibrationConvergence(
                    tr, traceT=tvec, paramForObs=None, showLegend=False,
                    runValues=P["laneFrac"][rows],
                    runValueLabel=f"{name} target / reference",
                    title=f"Calibration convergence -- parameters ({scope}) — {m} ladder on {name}")
                plt.show()
# endregion

## Run timings

In [ ]:
# region -> batched timing summary per ladder mode (total wall + amortized per-lane)
if runConfig["plot"] and pops:
    rows = []
    for m, P in pops.items():
        meta = P["timingMeta"] or {}
        total, n = meta.get("total_wall"), (meta.get("nrModels") or P["N"])
        rows.append({"mode": m, "device": meta.get("device", "?"),
                     "precision": meta.get("precision", "?"), "solver": meta.get("solver", "?"),
                     "dt": meta.get("dt"), "runTime": meta.get("runTime"), "nrModels": n,
                     "total_wall_s": total,
                     "amortized_s/lane": (total / n) if (total and n) else None})
    pd.DataFrame(rows)
# endregion

## Residual + parameter elasticity along each ladder

<details>
<summary>Per-lane residual against its own target, and the elasticity of every calibrated parameter.</summary>

For calibrated parameter $p$ and ladder fraction $f$, the elasticity is the slope of a
least-squares fit of $\ln p$ on $\ln f$ over the converged rungs. Because the laddered target is
$y^{*} = f \cdot y_{ref}$, that slope is exactly
$\varepsilon = \partial \ln p / \partial \ln y^{*}$ — the dimensionless bias factor the
reviewer asked for, in whatever units that target carries. $|\varepsilon| \approx 0$ means the
inferred parameter is indifferent to the assumed target; $|\varepsilon| \approx 1$ means it
inherits the assumption's error one-for-one.

The slack panel reads the uncontrolled venous volume at run end. When a volume target is
laddered it is forced to move (that is the conservation constraint doing its job), which is the
concrete demonstration that the published partition was never truly held fixed; under a
pressure or flow ladder it shows how much volume redistribution the recalibration implies.
</details>

In [ ]:
# region -> per-lane residual + d ln(param)/d ln(target) elasticity per ladder
if runConfig["plot"] and pops:
    def _resid(obsM, tgtM):
        """(maxRel%, rmsRel%) per lane over the targeted observations, vs each lane's OWN target."""
        rel = (obsM[:, targeted] - tgtM[:, targeted]) / tgtM[:, targeted] * 100.0
        return np.nanmax(np.abs(rel), axis=1), np.sqrt(np.nanmean(rel ** 2, axis=1))

    def _elasticity(y, f):
        """Slope of ln(y) on ln(f) — the dimensionless d ln(param)/d ln(target)."""
        ok = np.isfinite(y) & (y > 0) & np.isfinite(f) & (f > 0)
        if ok.sum() < 3:
            return np.nan
        return float(np.polyfit(np.log(f[ok]), np.log(y[ok]), 1)[0])

    ladders, slackByLadder, elastCols, residByLane = {}, {}, {}, {}
    for m, P in pops.items():
        rMax, rRMS = _resid(P["obsMatrix"], P["targetMatrix"])
        residByLane[m] = (rMax, rRMS)
        for name in dict.fromkeys(P["laneLadder"]):
            sel = (P["laneLadder"] == name) & P["finite"]
            if sel.sum() == 0:
                continue
            f = P["laneFrac"][sel]
            ladders[name]       = {"frac": f, "max": rMax[sel], "rms": rRMS[sel]}
            slackByLadder[name] = P["slack"][sel]
            elastCols[name]     = {p: _elasticity(P["lastRow"][sel, P["sigIdx"][p]], f)
                                   for p in P["plotParams"]}

    paramsInTable = [p for p in calibParams if any(p in c for c in elastCols.values())]
    ladderNames   = list(ladders)
    elastMatrix   = np.array([[elastCols[l].get(p, np.nan) for l in ladderNames]
                              for p in paramsInTable], dtype=float)

    elastDF = pd.DataFrame(elastMatrix, index=paramsInTable, columns=ladderNames)
    for name in ladderNames:
        d = ladders[name]
        print(f"[{name}] {len(d['frac'])} converged rungs over "
              f"f in [{d['frac'].min():.2f}, {d['frac'].max():.2f}] | "
              f"residual max|rel| {np.nanmin(d['max']):.3g}–{np.nanmax(d['max']):.3g}% | "
              f"slack {np.nanmin(slackByLadder[name]):.4g}–{np.nanmax(slackByLadder[name]):.4g} mL")
    print("\nelasticity  d ln(param) / d ln(target):")
    elastDF.round(3)
# endregion

## Figure — parameter elasticity across the ladders

In [ ]:
# region -> figure: elasticity heatmap (calibrated params x laddered targets)
if runConfig["plot"] and pops and ladders:
    fig = libPlots.plotVolumeLadder(
        elastMatrix, utils.labelsFor(paramsInTable, "latex"),
        ladderNames=[utils.labelFor(n, "latex", default=n) for n in ladderNames],
        title="Sensitivity of the inferred parameters to each calibration target")

    paperF = runConfig.get("paper", {})
    if paperF.get("emit", False) and fig is not None:
        imgDir = paperF.get("imagesDir", "EFC_Paper/revision/Images")
        os.makedirs(imgDir, exist_ok=True)
        outFig = os.path.join(imgDir, paperF.get("figName", "volumeLadder.png"))
        fig.savefig(outFig, dpi=200, bbox_inches="tight")
        print(f"figure -> {outFig}")
    plt.show()
# endregion

## LaTeX table — residual and worst-case parameter bias per ladder

<details>
<summary>One row per ladder: residual at the reference rung and at the extremes, plus the most strongly biased parameters.</summary>

Written to `paper.generatedDir/paper.tableName` when `paper.emit` is true. The `Bias` columns
read as "over the swept range of this target, the inferred parameter moved by this much" — the
direct answer to R1-C6 when the swept targets are the compartment volumes.
</details>

In [ ]:
# region -> LaTeX table: residual + most-biased params per ladder -> paper generatedDir
if runConfig["plot"] and pops and ladders:
    nP = runConfig["analysis"].get("nParamsInTable", 2)
    rowsTex = {}
    for name in ladderNames:
        d, f = ladders[name], ladders[name]["frac"]
        iRef, iLo, iHi = int(np.argmin(np.abs(f - 1.0))), int(np.argmin(f)), int(np.argmax(f))
        eps = {p: elastCols[name].get(p, np.nan) for p in paramsInTable}
        top = sorted((p for p in eps if np.isfinite(eps[p])),
                     key=lambda p: abs(eps[p]), reverse=True)[:nP]
        cells = [f"{f[iLo]:.2f}--{f[iHi]:.2f}",
                 f"{d['max'][iRef]:.3g}", f"{max(d['max'][iLo], d['max'][iHi]):.3g}",
                 f"{np.nanmin(slackByLadder[name]):.0f}--{np.nanmax(slackByLadder[name]):.0f}"]
        for p in top:
            cells.append(f"{utils.labelFor(p, 'latex', default=p)}: {eps[p]:+.2f}")
        cells += ["--"] * (nP - len(top))
        rowsTex[utils.labelFor(name, "latex", default=name.replace("_", r"\_"))] = cells

    cols = ["Ladder", r"Range ($\times y_{ref}$)", r"Res. at ref \%", r"Res. at extremes \%",
            r"Slack $V_{Vs}$ (mL)"] + [rf"Bias {i + 1} ($\varepsilon$)" for i in range(nP)]
    worstEps = np.nanmax(np.abs(elastMatrix)) if np.isfinite(elastMatrix).any() else float("nan")
    worstRes = max(float(np.nanmax(ladders[n]["max"])) for n in ladderNames)
    caption = (rf"Sensitivity of the calibrated parameter set to the imposed calibration "
               rf"targets, of which the fixed blood-volume partition is the case raised in "
               rf"review. Each ladder is a population of lanes that share the same initial state "
               rf"and the same sixteen physiological targets except one, which is stepped across "
               rf"the stated multiple of its reference value and recalibrated by the unchanged "
               rf"method. Bias columns give the elasticity "
               rf"$\varepsilon = \partial\ln p / \partial\ln y^{{*}}$ of the most strongly "
               rf"affected calibrated parameters, so a $10\%$ error in an assumed target "
               rf"shifts the inferred parameter by $10\varepsilon\%$. The largest elasticity "
               rf"observed is {worstEps:.2f} and the worst residual across all ladders is "
               rf"{worstRes:.2g}\%; the systemic venous compartment is uncontrolled and absorbs "
               rf"the residual volume, so its range shows how far the partition already moves "
               rf"under a conserved total blood volume.")
    table = utils.generate_latex_table_new(rowsTex, cols, "", "", "volumeLadder", caption)

    paperT = runConfig.get("paper", {})
    if paperT.get("emit", False):
        genDir = paperT.get("generatedDir", "EFC_Paper/revision/generated")
        os.makedirs(genDir, exist_ok=True)
        outTable = os.path.join(genDir, paperT.get("tableName", "volumeLadder.tex"))
        with open(outTable, "w") as fh:
            fh.write(table)
        print(f"table -> {outTable}")
    print(table)
# endregion